# Parsing netlist formats

`kfnetlist.morphnet` reads seven netlist formats into one schema document,
the morphnet `Netlist` (circuits, subcircuit instances, connections,
parameters, directives). `morphnet.to_netlists()` then turns every circuit
into a `kfnetlist.Netlist` connectivity graph.

| Format | Parser | Writer | Typical extension |
|---|---|---|---|
| SPICE (ngspice / generic) | `parse_spice` | `write_spice` | `.sp`, `.cir` |
| HSPICE | `parse_hspice` | `write_hspice` | `.sp` |
| Xyce | `parse_xyce` | `write_xyce` | `.cir` |
| Spectre | `parse_spectre` | `write_spectre` | `.scs` |
| Spectre with `simulator lang=spice` blocks | `parse_spectre_spice` | `write_spectre_spice` | `.scs` |
| Verilog-AMS | `parse_vams` | `write_vams` | `.vams`, `.va` |
| SystemVerilog (structural) | `parse_systemverilog` | `write_systemverilog` | `.sv` |

This page defines one example netlist per format, parses it, and shows
what ends up in the morphnet `Netlist` and in `kfnetlist.Netlist`. The
[conversion page](format_conversion.py) writes them back out in other formats.

**Waveguide or electrical?** Every parser applies the same rule to terminal
names: `o1`, `o2`, ... are waveguide (optical) terminals, while `e1`, `e2`,
... and names without a prefix are electrical. A net that touches an `o<N>`
terminal is a waveguide net.

In [1]:
from collections import defaultdict

from kfnetlist import NetlistPort, morphnet
from kfnetlist.morphnet.netlist import (
    parse_hspice,
    parse_spectre,
    parse_spectre_spice,
    parse_spice,
    parse_systemverilog,
    parse_vams,
    parse_xyce,
)
from kfnetlist.morphnet.netlist.value_utils import format_si_value


def value(param: morphnet.Parameter) -> str:
    pv = param.default_value
    if pv is not None and pv.prefixed_value is not None:
        return format_si_value(pv.prefixed_value)
    return param.properties.get("expr", "?")


def summary(netlist: morphnet.Netlist) -> None:
    """Print what a parser put into a morphnet Netlist."""
    print("top circuit:", netlist.top_circuit)
    for d in netlist.directives:
        print("directive:", d.kind.name, " ".join(filter(None, (d.name, d.value))))
    for ext in netlist.ext_circuits:
        ports = ", ".join(p.name for p in ext.ports)
        print(f"external circuit: {ext.name}({ports}) {ext.properties}")
    for c in netlist.circuits:
        print(f"\ncircuit {c.name}")
        if c.parameters:
            print(
                "  parameters:", ", ".join(f"{p.name}={value(p)}" for p in c.parameters)
            )
        print(
            "  ports:",
            ", ".join(
                f"{p.name} ({p.direction.name.lower()}, {p.domain.name.lower()})"
                for p in c.ports
            ),
        )
        for s in c.subcircuits:
            params = " ".join(f"{p.name}={value(p)}" for p in s.parameters)
            model = s.properties.get("model_name", "")
            print(f"  instance {s.name}: {s.circuit_name} {model} {params}".rstrip())
        nets: dict[str, list[str]] = defaultdict(list)
        domains = {}
        for conn in c.connections:
            for ref in (conn.source, conn.target):
                if ref is None:
                    continue
                pin = (
                    f"{ref.instance_name}.{ref.port_name}"
                    if ref.instance_name
                    else ref.port_name
                )
                if pin not in nets[conn.name]:
                    nets[conn.name].append(pin)
            domains[conn.name] = conn.domain.name.lower()
        for name, pins in nets.items():
            print(f"  net {name} [{domains[name]}]: {', '.join(pins)}")
        for key, text in c.properties.items():
            print(f"  {key}: {' '.join(text.split())}")


def show_kfnetlist(netlist: morphnet.Netlist, cell: str) -> None:
    """Print the kfnetlist.Netlist that morphnet.to_netlists() builds for `cell`."""
    nl = morphnet.to_netlists(netlist)[cell]
    nl.sort()
    print("ports:", [p.name for p in nl.ports])
    for name, inst in nl.instances.items():
        print(f"instance {name}: component={inst.component!r} settings={inst.settings}")
    for net in nl.nets:
        print(
            "net:",
            " -- ".join(
                m.name if isinstance(m, NetlistPort) else f"{m.instance}.{m.port}"
                for m in net
            ),
        )

## SPICE

A generic SPICE deck: device models, a CMOS inverter subcircuit, and a
buffer built from two inverters. Primitive devices (`R`, `C`, `L`, `D`,
`Q`, `M`, `V`, `I`) become instances of external circuits (`resistor`,
`mosfet`, ...) with fixed pin names (`p`/`n`, `d`/`g`/`s`/`b`, ...). The
model name is kept in the instance's `model_name` property.

In [2]:
SPICE = """\
* CMOS buffer
.model nch nmos level=49 vth0=0.4
.model pch pmos level=49 vth0=-0.4

.subckt inverter in out vdd vss
M1 out in vdd vdd pch W=2u L=180n
M2 out in vss vss nch W=1u L=180n
.ends inverter

.subckt buffer a y vdd vss
X1 a mid vdd vss inverter
X2 mid y vdd vss inverter
C1 y vss 10f
.ends buffer
.end
"""
spice = parse_spice(SPICE)
summary(spice)

top circuit: buffer
external circuit: mosfet(d, g, s, b) {'spice_prefix': 'M'}
external circuit: capacitor(p, n) {'spice_prefix': 'C'}
external circuit: nch() {'model_type': 'nmos'}
external circuit: pch() {'model_type': 'pmos'}

circuit inverter
  ports: in (inout, electrical), out (inout, electrical), vdd (inout, electrical), vss (inout, electrical)
  instance M1: mosfet pch W=2u L=180n
  instance M2: mosfet nch W=1u L=180n
  net in [electrical]: in, M1.g, M2.g
  net out [electrical]: out, M1.d, M2.d
  net vdd [electrical]: vdd, M1.s, M1.b
  net vss [electrical]: vss, M2.s, M2.b

circuit buffer
  ports: a (inout, electrical), y (inout, electrical), vdd (inout, electrical), vss (inout, electrical)
  instance X1: inverter
  instance X2: inverter
  instance C1: capacitor  value=10f
  net a [electrical]: a, X1.in
  net y [electrical]: y, X2.out, C1.p
  net vdd [electrical]: vdd, X1.vdd, X2.vdd
  net vss [electrical]: vss, X1.vss, X2.vss, C1.n
  net mid [electrical]: X1.out, X2.in


`to_netlists()` returns one `kfnetlist.Netlist` per circuit. Instances keep
their parameters as `settings`; SI values become plain floats:

In [3]:
show_kfnetlist(spice, "inverter")

ports: ['in', 'out', 'vdd', 'vss']
instance M1: component='mosfet' settings={'L': 1.8000000000000002e-07, 'W': 2e-06}
instance M2: component='mosfet' settings={'L': 1.8000000000000002e-07, 'W': 1e-06}
net: in -- M1.g -- M2.g
net: out -- M1.d -- M2.d
net: vdd -- M1.b -- M1.s
net: vss -- M2.b -- M2.s


In [4]:
show_kfnetlist(spice, "buffer")

ports: ['a', 'vdd', 'vss', 'y']
instance C1: component='capacitor' settings={'value': 1.0000000000000002e-14}
instance X1: component='inverter' settings={}
instance X2: component='inverter' settings={}
net: a -- X1.in
net: vdd -- X1.vdd -- X2.vdd
net: vss -- C1.n -- X1.vss -- X2.vss
net: y -- C1.p -- X2.out
net: X1.out -- X2.in


## HSPICE

HSPICE is parsed with the SPICE grammar plus HSPICE preprocessing (for
example, `$` comments). Here is an RC low-pass filter with an HSPICE inline
comment:

In [5]:
HSPICE = """\
* RC low-pass
.subckt rc_lowpass in_ out gnd
R1 in_ out 1k     $ series resistor
C1 out gnd 100p   $ shunt capacitor
.ends rc_lowpass
.end
"""
hspice = parse_hspice(HSPICE)
summary(hspice)

top circuit: rc_lowpass
external circuit: resistor(p, n) {'spice_prefix': 'R'}
external circuit: capacitor(p, n) {'spice_prefix': 'C'}

circuit rc_lowpass
  ports: in_ (inout, electrical), out (inout, electrical), gnd (inout, electrical)
  instance R1: resistor  value=1k
  instance C1: capacitor  value=100p
  net in_ [electrical]: in_, R1.p
  net out [electrical]: out, R1.n, C1.p
  net gnd [electrical]: gnd, C1.n


In [6]:
show_kfnetlist(hspice, "rc_lowpass")

ports: ['gnd', 'in_', 'out']
instance C1: component='capacitor' settings={'value': 1e-10}
instance R1: component='resistor' settings={'value': 1000.0}
net: gnd -- C1.n
net: in_ -- R1.p
net: out -- C1.p -- R1.n


## Xyce

Xyce decks are case-insensitive and declare subcircuit parameters with
`PARAMS:`. A resistive divider, where each resistor value is a subcircuit
parameter:

In [7]:
XYCE = """\
* Xyce divider
.SUBCKT divider vin vout gnd PARAMS: rtop=10k rbot=10k
R1 vin vout 10k
R2 vout gnd 10k
.ENDS divider
.END
"""
xyce = parse_xyce(XYCE)
summary(xyce)

top circuit: divider
external circuit: resistor(p, n) {'spice_prefix': 'R'}

circuit divider
  parameters: rtop=10k, rbot=10k
  ports: vin (inout, electrical), vout (inout, electrical), gnd (inout, electrical)
  instance R1: resistor  value=10k
  instance R2: resistor  value=10k
  net vin [electrical]: vin, R1.p
  net vout [electrical]: vout, R1.n, R2.p
  net gnd [electrical]: gnd, R2.n


In [8]:
show_kfnetlist(xyce, "divider")

ports: ['gnd', 'vin', 'vout']
instance R1: component='resistor' settings={'value': 10000.0}
instance R2: component='resistor' settings={'value': 10000.0}
net: gnd -- R2.n
net: vin -- R1.p
net: vout -- R1.n -- R2.p


## Spectre

Spectre puts terminals in parentheses and names the master after them.
A two-stage hierarchy:

In [9]:
SPECTRE = """\
subckt inverter (in_ out vdd vss)
MP (out in_ vdd vdd) pch w=2u l=180n
MN (out in_ vss vss) nch w=1u l=180n
ends inverter

subckt buffer (a y vdd vss)
I1 (a mid vdd vss) inverter
I2 (mid y vdd vss) inverter
ends buffer
"""
spectre = parse_spectre(SPECTRE)
summary(spectre)

top circuit: buffer
external circuit: mosfet(d, g, s, b) {'spice_prefix': 'M'}

circuit inverter
  ports: in_ (inout, electrical), out (inout, electrical), vdd (inout, electrical), vss (inout, electrical)
  instance MP: mosfet pch w=2u l=180n
  instance MN: mosfet nch w=1u l=180n
  net in_ [electrical]: in_, MP.g, MN.g
  net out [electrical]: out, MP.d, MN.d
  net vdd [electrical]: vdd, MP.s, MP.b
  net vss [electrical]: vss, MN.s, MN.b

circuit buffer
  ports: a (inout, electrical), y (inout, electrical), vdd (inout, electrical), vss (inout, electrical)
  instance I1: inverter
  instance I2: inverter
  net a [electrical]: a, I1.in_
  net y [electrical]: y, I2.out
  net vdd [electrical]: vdd, I1.vdd, I2.vdd
  net vss [electrical]: vss, I1.vss, I2.vss
  net mid [electrical]: I1.out, I2.in_


In [10]:
show_kfnetlist(spectre, "buffer")

ports: ['a', 'vdd', 'vss', 'y']
instance I1: component='inverter' settings={}
instance I2: component='inverter' settings={}
net: a -- I1.in_
net: vdd -- I1.vdd -- I2.vdd
net: vss -- I1.vss -- I2.vss
net: y -- I2.out
net: I1.out -- I2.in_


## Spectre with SPICE blocks

Spectre decks often switch language with `simulator lang=...`. The
`spectre_spice` parser handles both halves in one file:

In [11]:
SPECTRE_SPICE = """\
simulator lang=spice
.subckt rdiv a b mid
R1 a mid 1k
R2 mid b 1k
.ends rdiv

simulator lang=spectre
subckt top (vin gnd)
X0 (vin gnd tap) rdiv
C0 (tap gnd) capacitor c=1p
ends top
"""
spectre_spice = parse_spectre_spice(SPECTRE_SPICE)
summary(spectre_spice)

top circuit: top
external circuit: resistor(p, n) {'spice_prefix': 'R'}
external circuit: capacitor(p, n) {'spice_prefix': 'C'}

circuit rdiv
  ports: a (inout, electrical), b (inout, electrical), mid (inout, electrical)
  instance R1: resistor  value=1k
  instance R2: resistor  value=1k
  net a [electrical]: a, R1.p
  net b [electrical]: b, R2.n
  net mid [electrical]: mid, R1.n, R2.p

circuit top
  ports: vin (inout, electrical), gnd (inout, electrical)
  instance X0: rdiv
  instance C0: capacitor  c=1p
  net vin [electrical]: vin, X0.a
  net gnd [electrical]: gnd, X0.b, C0.n
  net tap [electrical]: X0.mid, C0.p


In [12]:
show_kfnetlist(spectre_spice, "top")

ports: ['gnd', 'vin']
instance C0: component='capacitor' settings={'c': 1e-12}
instance X0: component='rdiv' settings={}
net: gnd -- C0.n -- X0.b
net: vin -- X0.a
net: C0.p -- X0.mid


## Verilog-AMS

Verilog-AMS modules declare port directions and disciplines (the discipline
is kept as `Port.discipline`). `analog` blocks are not structure: they are
kept as text in the circuit's `analog_block` property.

In [13]:
VAMS = """\
`include "disciplines.vams"
module res_model (p, n);
  inout p, n;
  electrical p, n;
  parameter real r = 1k;
  analog V(p, n) <+ r * I(p, n);
endmodule

module divider (vin, vout, gnd);
  input vin;
  output vout;
  inout gnd;
  electrical vin, vout, gnd;
  res_model #(.r(10k)) R1 (.p(vin), .n(vout));
  res_model #(.r(10k)) R2 (.p(vout), .n(gnd));
endmodule
"""
vams = parse_vams(VAMS)
summary(vams)

top circuit: divider
directive: INCLUDE disciplines.vams

circuit res_model
  parameters: r=1k
  ports: p (inout, electrical), n (inout, electrical)
  analog_block: analog V(p, n) <+ r * I(p, n);

circuit divider
  ports: vin (input, electrical), vout (output, electrical), gnd (inout, electrical)
  instance R1: res_model  r=10k
  instance R2: res_model  r=10k
  net vin [electrical]: vin, R1.p
  net vout [electrical]: vout, R1.n, R2.p
  net gnd [electrical]: gnd, R2.n


In [14]:
show_kfnetlist(vams, "divider")

ports: ['gnd', 'vin', 'vout']
instance R1: component='res_model' settings={'r': 10000.0}
instance R2: component='res_model' settings={'r': 10000.0}
net: gnd -- R2.n
net: vin -- R1.p
net: vout -- R1.n -- R2.p


## SystemVerilog

The SystemVerilog parser reads the structural part of the language
(SystemVerilog 3.1a, Annex A):

- ANSI headers `(input wire [3:0] a, output y)` and non-ANSI port lists with
  body declarations;
- `parameter`/`localparam`, also in the `#(...)` header; numbers, sized
  literals (`4'b1010`) and expressions (kept as text in an `expr` property);
- instances with named `.p(n)`, positional, implicit `.p`, wildcard `.*` and
  empty `.p()` connections, several per statement;
- `assign a = b;` and `alias a = b = c;`, which merge nets.

Behavioral code (`always`, `initial`, `function`, `task`, `generate`,
`class`, `assign` with logic on the right, ...) is kept as text in
`behavioral_block_N` properties.

In [15]:
SYSTEMVERILOG = """\
`timescale 1ns/1ps
module nand2 (input a, b, output y, inout vdd, vss);
  pmos_t P1 (.d(y), .g(a), .s(vdd)), P2 (.d(y), .g(b), .s(vdd));
  nmos_t N1 (.d(y), .g(a), .s(mid)), N2 (.d(mid), .g(b), .s(vss));
endmodule

module latch (s_n, r_n, q, q_n, vdd, vss);
  input s_n, r_n;
  output q, q_n;
  inout vdd, vss;
  wire qi;
  nand2 U1 (.a(s_n), .b(q_n), .y(qi), .*);
  nand2 U2 (.a(r_n), .b(q), .y(q_n), .*);
  assign q = qi;
  initial $display("latch ready");
endmodule
"""
systemverilog = parse_systemverilog(SYSTEMVERILOG)
summary(systemverilog)

top circuit: latch
directive: TIMESCALE 1ns/1ps

circuit nand2
  ports: a (input, electrical), b (input, electrical), y (output, electrical), vdd (inout, electrical), vss (inout, electrical)
  instance P1: pmos_t
  instance P2: pmos_t
  instance N1: nmos_t
  instance N2: nmos_t
  net a [electrical]: a, P1.g, N1.g
  net b [electrical]: b, P2.g, N2.g
  net y [electrical]: y, P1.d, P2.d, N1.d
  net vdd [electrical]: vdd, P1.s, P2.s
  net vss [electrical]: vss, N2.s
  net mid [electrical]: N1.s, N2.d

circuit latch
  ports: s_n (input, electrical), r_n (input, electrical), q (output, electrical), q_n (output, electrical), vdd (inout, electrical), vss (inout, electrical)
  instance U1: nand2
  instance U2: nand2
  net s_n [electrical]: s_n, U1.a
  net r_n [electrical]: r_n, U2.a
  net q [electrical]: q, U1.y, U2.b
  net q_n [electrical]: q_n, U1.b, U2.y
  net vdd [electrical]: vdd, U1.vdd, U2.vdd
  net vss [electrical]: vss, U1.vss, U2.vss
  behavioral_block_0: initial $display("latch ready

In [16]:
show_kfnetlist(systemverilog, "latch")

ports: ['q', 'q_n', 'r_n', 's_n', 'vdd', 'vss']
instance U1: component='nand2' settings={}
instance U2: component='nand2' settings={}
net: q -- U1.y -- U2.b
net: q_n -- U1.b -- U2.y
net: r_n -- U2.a
net: s_n -- U1.a
net: vdd -- U1.vdd -- U2.vdd
net: vss -- U1.vss -- U2.vss


`assign q = qi;` merged `qi` into the port net `q`, and `.*` connected `vdd`
and `vss` by name.

## Photonic circuits

The same parsers read photonic netlists. Terminals named `o<N>` are
waveguide terminals; heater and bias pins (`e1`, `vdd`, ...) stay
electrical. A Mach-Zehnder interferometer with a thermal phase shifter, in
SystemVerilog:

In [17]:
MZI_SV = """\
module mzi #(parameter real arm_length = 100e-6) (
  input o1, o2, output o3, o4,   // optical
  input e1, inout gnd            // heater drive
);
  dc #(.gap(200n)) Xdc1 (.o1(o1), .o2(o2), .o3(top_a), .o4(bot_a));
  dc #(.gap(200n)) Xdc2 (.o1(top_b), .o2(bot_b), .o3(o3), .o4(o4));
  heater_wg #(.length(arm_length)) Xtop (.o1(top_a), .o2(top_b), .e1(e1), .e2(gnd));
  wg #(.length(arm_length)) Xbot (.o1(bot_a), .o2(bot_b));
endmodule
"""
mzi_sv = parse_systemverilog(MZI_SV)
summary(mzi_sv)

top circuit: mzi

circuit mzi
  parameters: arm_length=0.0001
  ports: o1 (input, waveguide), o2 (input, waveguide), o3 (output, waveguide), o4 (output, waveguide), e1 (input, electrical), gnd (inout, electrical)
  instance Xdc1: dc  gap=200n
  instance Xdc2: dc  gap=200n
  instance Xtop: heater_wg  length=arm_length
  instance Xbot: wg  length=arm_length
  net o1 [waveguide]: o1, Xdc1.o1
  net o2 [waveguide]: o2, Xdc1.o2
  net o3 [waveguide]: o3, Xdc2.o3
  net o4 [waveguide]: o4, Xdc2.o4
  net e1 [electrical]: e1, Xtop.e1
  net gnd [electrical]: gnd, Xtop.e2
  net top_a [waveguide]: Xdc1.o3, Xtop.o1
  net bot_a [waveguide]: Xdc1.o4, Xbot.o1
  net top_b [waveguide]: Xdc2.o1, Xtop.o2
  net bot_b [waveguide]: Xdc2.o2, Xbot.o2


The same MZI as a SPICE subcircuit. SPICE has no port names on subcircuit
calls, so the instance pins of undeclared subcircuits are numbered and only
the nets that reach an `o<N>` terminal of `mzi` itself are waveguide. Define the
subcircuits in the deck (as here) and every net keeps its domain:

In [18]:
MZI_SPICE = """\
.subckt dc o1 o2 o3 o4 gap=200n
.ends dc
.subckt wg o1 o2 length=100u
.ends wg
.subckt heater_wg o1 o2 e1 e2 length=100u
.ends heater_wg
.subckt mzi o1 o2 o3 o4 e1 gnd
Xdc1 o1 o2 top_a bot_a dc
Xdc2 top_b bot_b o3 o4 dc
Xtop top_a top_b e1 gnd heater_wg
Xbot bot_a bot_b wg
.ends mzi
"""
mzi_spice = parse_spice(MZI_SPICE)
summary(mzi_spice)

top circuit: mzi

circuit dc
  parameters: gap=200n
  ports: o1 (inout, waveguide), o2 (inout, waveguide), o3 (inout, waveguide), o4 (inout, waveguide)

circuit wg
  parameters: length=100u
  ports: o1 (inout, waveguide), o2 (inout, waveguide)

circuit heater_wg
  parameters: length=100u
  ports: o1 (inout, waveguide), o2 (inout, waveguide), e1 (inout, electrical), e2 (inout, electrical)

circuit mzi
  ports: o1 (inout, waveguide), o2 (inout, waveguide), o3 (inout, waveguide), o4 (inout, waveguide), e1 (inout, electrical), gnd (inout, electrical)
  instance Xdc1: dc
  instance Xdc2: dc
  instance Xtop: heater_wg
  instance Xbot: wg
  net o1 [waveguide]: o1, Xdc1.o1
  net o2 [waveguide]: o2, Xdc1.o2
  net o3 [waveguide]: o3, Xdc2.o3
  net o4 [waveguide]: o4, Xdc2.o4
  net e1 [electrical]: e1, Xtop.e1
  net gnd [electrical]: gnd, Xtop.e2
  net top_a [waveguide]: Xdc1.o3, Xtop.o1
  net bot_a [waveguide]: Xdc1.o4, Xbot.o1
  net top_b [waveguide]: Xdc2.o1, Xtop.o2
  net bot_b [waveguide]: X

Both describe the same graph:

In [19]:
show_kfnetlist(mzi_sv, "mzi")

ports:

 ['e1', 'gnd', 'o1', 'o2', 'o3', 'o4']
instance Xbot: component='wg' settings={}
instance Xdc1: component='dc' settings={'gap': 2.0000000000000002e-07}
instance Xdc2: component='dc' settings={'gap': 2.0000000000000002e-07}
instance Xtop: component='heater_wg' settings={}
net: e1 -- Xtop.e1
net: gnd -- Xtop.e2
net: o1 -- Xdc1.o1
net: o2 -- Xdc1.o2
net: o3 -- Xdc2.o3
net: o4 -- Xdc2.o4
net: Xbot.o1 -- Xdc1.o4
net: Xbot.o2 -- Xdc2.o2
net: Xdc1.o3 -- Xtop.o1
net: Xdc2.o1 -- Xtop.o2


Only the `o<N>` pattern counts: `out`, `osc` or `opt` stay electrical.

In [20]:
from kfnetlist.morphnet.netlist.net_utils import domain_of

for name in ("o1", "o12", "e1", "out", "osc", "vdd"):
    print(f"{name:>4}: {domain_of(name).name}")

  o1: WAVEGUIDE
 o12: WAVEGUIDE
  e1: ELECTRICAL
 out: ELECTRICAL
 osc: ELECTRICAL
 vdd: ELECTRICAL


## Defining a netlist in Python

Netlists can also be built directly with the morphnet builder API (see
[Basic Usage](basic-usage.md)), and then written in any format:

In [21]:
from kfnetlist.morphnet.netlist import write_systemverilog

WG = morphnet.SignalDomain.WAVEGUIDE
ring = morphnet.circuit("ring_filter", parameters=[("radius", 10e-6)])
for name in ("o1", "o2", "o3", "o4"):
    ring.add_terminal(name, domain=WG)
ring.add(morphnet.subcircuit("Xring", "ring_coupler", gap=150e-9))
for name in ("o1", "o2", "o3", "o4"):
    ring.connect(name, ("", name), ("Xring", name), domain=WG)

doc = morphnet.Netlist(name="ring", top_circuit="ring_filter").add(ring)
summary(doc)
print()
print(write_systemverilog(doc))

top circuit: ring_filter

circuit ring_filter
  parameters: radius=1e-05
  ports: o1 (inout, waveguide), o2 (inout, waveguide), o3 (inout, waveguide), o4 (inout, waveguide)
  instance Xring: ring_coupler  gap=1.5e-07
  net o1 [waveguide]: o1, Xring.o1
  net o2 [waveguide]: o2, Xring.o2
  net o3 [waveguide]: o3, Xring.o3
  net o4 [waveguide]: o4, Xring.o4

module ring_filter #(parameter radius = 1e-05) (
  inout o1,
  inout o2,
  inout o3,
  inout o4
);
  ring_coupler #(.gap(1.5e-07)) Xring (.o1(o1), .o2(o2), .o3(o3), .o4(o4));
endmodule

